In [20]:
import os
import shutil
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset, DatasetDict
from datasets import load_from_disk
from collections import defaultdict

In [21]:
input_dir = 'lct_txt_half'
output_dir = 'lct_p3_all_entities'
dataset_dir = 'dataset_p4_prompt2_new'

train_input_dir = os.path.join(dataset_dir, 'train', 'input')
train_output_dir = os.path.join(dataset_dir, 'train', 'output')
test_input_dir = os.path.join(dataset_dir, 'test', 'input')
test_output_dir = os.path.join(dataset_dir, 'test', 'output')

os.makedirs(train_input_dir, exist_ok=True)
os.makedirs(train_output_dir, exist_ok=True)
os.makedirs(test_input_dir, exist_ok=True)
os.makedirs(test_output_dir, exist_ok=True)

input_files = [f for f in input_files if f in output_files]

file_groups = defaultdict(list)
for f in input_files:
    nct_number = f.split('_')[0]
    file_groups[nct_number].append(f)


all_groups = list(file_groups.values())
train_groups, test_groups = train_test_split(all_groups, test_size=0.2, random_state=42)

train_files = [file for group in train_groups for file in group]
test_files = [file for group in test_groups for file in group]

In [22]:
def copy_files(file_list, src_dir, dest_dir):
    for file_name in file_list:
        shutil.copy(os.path.join(src_dir, file_name), os.path.join(dest_dir, file_name))

# Trainingsdaten kopieren
copy_files(train_files, input_dir, train_input_dir)
copy_files(train_files, output_dir, train_output_dir)

# Testdaten kopieren
copy_files(test_files, input_dir, test_input_dir)
copy_files(test_files, output_dir, test_output_dir)
print(f"Dataset creation completed. Training files: {len(train_files)}, Test files: {len(test_files)}")

Dataset creation completed. Training files: 1608, Test files: 404


In [24]:
entity_prompt2 = """ You are designed to parse textual clinical study eligibility criteria into a structured JSON format following the Composite pattern.
The input consists of raw text containing the suitability criteria, which are to be logically broken down into a tree structure using the logical operators "AND", "OR" and "NOT".
Provide a JSON response based on the following instructions:

1. Split the Text based on the logical operators AND, OR, NOT by using the following rules:
- Always insert an [OR] before an "or", "and/or", "and / or" in the text.
- Always insert an [OR] after a comma "," and after "/" in the text.
- Always insert an [AND] before words like "with", "who", "in addition", "plus", "and", "but", "that", "despite", "having".
- Always insert a [NOT] before "no", "not", "none",  "don't", "free", "prevent", "Inability", "lack", "impossible", "off", "without", "unable", "naive", "excluded", "absence"
- Always insert a [AND] before a [NOT].
- Always insert a [AND] at the end of a sentence if another sentence follows.

2. Create a JSON object based on the following rules:
- The basic structure consists of nested "AND", "OR" and "NOT" operators.
- Each AND and OR operator always has two branches: "left" and "right". Every left must always have a corresponding right.
- The NOT operator has only one branch: "left".
- "raw_text"  are leaf nodes with no children.
- The extracted "raw_text" segments should be connected using the operators "AND", "OR", and "NOT" as per the rules above.

3. Sequence and structure:
- AND/OR always start with "left" and "right" branches
- NOT always starts with a "left" branch
- A "left" or "right" key is followed by either: another operator (AND, OR, NOT) or an object with "raw_text" (and optional additional fields)
- "raw_text" fields contain the actual text of the criterion. The raw text must always be transferred in full.
- Additional fields such as "Drug", "Condition", "Observation" or "Age"  can appear next to "raw_text" to categorise specific terms.

4. Extract the following entities from the raw text and include them in the JSON object:
Nodes could include attributes such as:
- "raw_text": A string representing the raw text description of the medical condition, procedure, or other relevant details.
- all entities: 'Contraindication', 'Eq-Value', 'Severity', 'Drug', 'Observation-Name', 'Age', 'Location', 'Organism-Name', 'Encounter', 'Drug-Name', 'Ethnicity', 'Modifier', 'Condition', 'Eq-Unit', 'Eq-Temporal-Unit', 'Family-Member', 'Organism', 'Other', 'Immunization-Name', 'Polarity', 'Condition-Type', 'Immunization', 'Eq-Operator', 'Eq-Temporal-Recency', 'Procedure-Name', 'Indication', 'Exception', 'Study', 'Language', 'Coreference', 'Provider', 'Acuteness', 'Life-Stage-And-Gender', 'Procedure', 'Risk', 'Death', 'Assertion', 'Allergy-Name', 'Specimen', 'Negation', 'Code', 'Stability', 'Birth', 'Criteria-Count', 'Eq-Comparison', 'Condition-Name', 'Insurance', 'Observation', 'Allergy', 'Eq-Temporal-Period'
- the lists for the entity's should only be appended if one or more entities have been found. Otherwise, omit the arrays completely.

5. The JSON object should be containing the following rules:
- Ensure that the same logical operator ("AND", "OR", "NOT") does not appear on the same level of the tree structure to maintain the integrity of the logical relationships.
- Generates JSON files from the textual criteria and organizing them into the described logical structure.
- Setting the brackets {}, [] and commas , correctly in the JSON is extremely important.
- Any double quotes within the raw textual criteria are escaped with a backslash (\").
- If there are no restrictions, return an empty JSON FILE {}
- The entire text of the eligibility criteria must be included and separated by the logical operators, without omitting any words or information.
- The output should only contain the JSON object, without any additional text or formatting.
- Ensure that the JSON output strictly follows these rules and accurately represents the logical structure of the input eligibility criteria."""

### Create Dataset

In [26]:
dataset_type = "dataset_p4_prompt2_new"

train_input_dir = f'{dataset_type}/train/input'
train_output_dir = f'{dataset_type}/train/output'
test_input_dir = f'{dataset_type}/test/input'
test_output_dir = f'{dataset_type}/test/output'


def create_dataframe(input_dir, output_dir):
    data = []
    for file_name in os.listdir(input_dir):
        if file_name in os.listdir(output_dir):
            with open(os.path.join(input_dir, file_name), 'r', encoding='utf-8') as f_in, \
                    open(os.path.join(output_dir, file_name), 'r', encoding='utf-8') as f_out:
                input_text = f_in.read().strip()
                output_text = f_out.read().strip()
                nct_number = os.path.splitext(file_name)[0]
                data.append({"input": input_text, "output": output_text, "nct_number": nct_number})
    return pd.DataFrame(data)

# DataFrames für Training und Test erstellen
train_df = create_dataframe(train_input_dir, train_output_dir)
test_df = create_dataframe(test_input_dir, test_output_dir)
train_df['instruction'] = entity_prompt2 # Ändere hier die Prompt bei der Erstellung
test_df['instruction'] = entity_prompt2
train_df = train_df[['input', 'output', 'instruction']] # , 'nct_number'
test_df = test_df[['input', 'output', 'instruction']] # , 'nct_number'
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

In [27]:
dataset_path = 'dataset_p4_prompt2_new/dataset_p4_prompt2'

dataset_dict.save_to_disk(dataset_path)
print("Datasets wurden erfolgreich erstellt und gespeichert.")

Saving the dataset (0/1 shards):   0%|          | 0/1608 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/404 [00:00<?, ? examples/s]

Datasets wurden erfolgreich erstellt und gespeichert.


In [28]:
dataset_dict = load_from_disk(dataset_path)
train_dataset = dataset_dict['train']

### Test: Train Test Eval

In [7]:
train_test_split = train_dataset.train_test_split(test_size=0.1, seed=42)

In [10]:
train_test_split

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 723
    })
    test: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 81
    })
})

In [7]:
dataset_dict

In [8]:
print("Train Dataset:")
print(dataset_dict['train'][0])

In [33]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

#EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) #+ EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

In [34]:
from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [37]:
dataset[0]

In [38]:

dataset_path = 'dataset/lct_dataset_v2'
dataset = load_from_disk(dataset_path)
dataset = dataset['train']
dataset = dataset.map(formatting_prompts_func, batched=True)